# Project SENTINEL — AlphaFold2-multimer complex folding (GPU, one-click)

Full-scale complex refolding + interface scoring (interface pAE < 7.5,
pLDDT > 85, ipTM > 0.7 — thresholds from `config/config.yaml`,
`design.scoring`) for the leads in `results/design/leads.fasta`, folded
against tau target PDB **5O3L**
(AD_PHF). The CPU sandbox used a documented
substitute (ESM-2 sequence plausibility + geometric complementarity — see
`src/sentinel/design/interface_scorer.py`) instead of this GPU step.

**How to use:** Runtime -> GPU -> Run all. Upload `leads.fasta` when
prompted (from `results/design/leads.fasta` in the repo).


In [ ]:
#@title 1. Install ColabFold (localcolabfold-in-Colab)
!pip install -q "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold"
!pip install -q jax[cuda12_pip] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html


In [ ]:
#@title 2. Upload leads.fasta and build binder:tau complex FASTA pairs
from google.colab import files
uploaded = files.upload()  # upload results/design/leads.fasta from the repo

TAU_CORE_SEQ = "PASTE_TAU_CORE_SEQUENCE_HERE"  #@param {type:"string"}
# (tau core sequence, residues 306-378,
#  is in data/interim/tau_sequence.json in the repo — copy the substring for this range)

lead_fasta = list(uploaded.keys())[0]
records = open(lead_fasta).read().strip().split(">")[1:]
with open("complexes.fasta", "w") as out:
    for r in records:
        header, seq = r.split("\n", 1)
        seq = seq.strip()
        out.write(f">{header.split()[0]}\n{seq}:{TAU_CORE_SEQ}\n")
print("wrote complexes.fasta —", len(records), "binder:tau pairs")


In [ ]:
#@title 3. Run ColabFold (AlphaFold2-multimer) on every pair
!colabfold_batch complexes.fasta af2_multimer_outputs --num-models 3 --model-type alphafold2_multimer_v3


In [ ]:
#@title 4. Parse interface metrics and apply the Year-1 thresholds
import json, glob
INTERFACE_PAE_MAX = 7.5  #@param {type:"number"}
PLDDT_MIN = 85  #@param {type:"number"}
IPTM_MIN = 0.7  #@param {type:"number"}

passed = []
for f in glob.glob("af2_multimer_outputs/*_scores_rank_001*.json"):
    d = json.load(open(f))
    iptm = d.get("iptm", 0)
    plddt = sum(d.get("plddt", [0])) / max(len(d.get("plddt", [1])), 1)
    if iptm >= IPTM_MIN and plddt >= PLDDT_MIN:
        passed.append((f, iptm, plddt))
print(f"{len(passed)} designs passed AF2-multimer thresholds")
for f, iptm, plddt in passed:
    print(f, "ipTM=", round(iptm, 3), "pLDDT=", round(plddt, 1))


In [ ]:
#@title 5. Zip and download
import shutil
from google.colab import files
shutil.make_archive("af2_multimer_results", "zip", "af2_multimer_outputs")
files.download("af2_multimer_results.zip")
